# 09 — Private Exchange Setup: Internal Data Marketplace

This notebook adds the **Databricks Marketplace Private Exchange** layer on top of the existing Data Mesh infrastructure (notebooks 00–08). It transforms how data products are **distributed and consumed** across the organization.

## The Shift: From Catalog Grants to a Marketplace Storefront

| Before (Direct UC Grants) | After (Private Exchange) |
|---|---|
| Admin manually runs `GRANT SELECT` on gold tables | Domain teams **publish** products to a private marketplace |
| Consumers must know exact catalog.schema.table names | Consumers **browse and search** a storefront with descriptions |
| No formal request/approval workflow | Self-service **Request → Approve → Install** lifecycle |
| Documentation limited to `COMMENT` and `TBLPROPERTIES` | Rich listing pages with use cases, sample queries, docs |

## End-to-End Flow: Domain Catalog → Private Exchange → Consumer

| Step | Component | Example |
|---|---|---|
| 1. Domain Catalog | Gold table owned by domain team | `adoit_product.gold.application_landscape` |
| 2. Delta Sharing Share | Logical grouping of tables to share | `adoit_application_landscape` |
| 3. Private Listing | Marketplace listing with rich metadata | "Application Landscape" (category, docs, samples) |
| 4. Private Exchange | Invite-only storefront for members | "Enterprise Data Mesh Exchange" |
| 5. Consumer | Business unit discovers & installs | CTO Office installs into their catalog |

## Prerequisites
- **Premium workspace** with Unity Catalog enabled
- **Marketplace admin role** assigned (Account Console → Users → Roles)
- Notebooks 00–08 executed (domain catalogs, gold tables, governance tables exist)
- Storage: `abfss://data-products@sadpsddbxdev.dfs.core.windows.net/`

## Step 1: Provider Signup (One-Time Setup)

Becoming a Private Exchange provider is a **one-time UI action** — it cannot be automated via SQL or API. Follow these steps in the Databricks workspace:

### Procedure
1. **Navigate to Marketplace** — Click the Marketplace icon (🏪) in the left sidebar
2. **Open Provider Console** — Click **"Provider Console"** in the upper-right corner of the Marketplace page
3. **Start provider signup** — The console displays the "Get started as a provider" page. Follow the self-service flow to enable your account as a **private-exchange-only provider**
4. **Assign Marketplace admin role** — After a few minutes, click "Assign Marketplace admin" which opens the Account Console. On the Roles tab, enable **Marketplace admin** for yourself (and any other admins)
5. **Return & refresh** — Go back to the Provider Console in the workspace and click **"Refresh page"** (not your browser's refresh). Wait a few minutes for the role to propagate
6. **Create Provider Profile** — Click **"Create profile"** and fill in:
   - **Provider name**: Your organization name (e.g., "Enterprise Data Platform")
   - **Logo**: Organization logo (light + dark mode)
   - **Description**: "Centralized Data Mesh platform providing governed data products across Enterprise Architecture, IT Service Management, and cross-domain analytics."
   - **Organization website**: Your internal portal URL
   - **Business email**: Platform team distribution list
   - **Support email**: Data product support channel
   - **Terms of service link**: Internal data governance policy URL
   - **Privacy policy link**: Internal data privacy policy URL

> ⚠️ **Note**: This step is performed once per Databricks account. After completion, you can create exchanges and listings programmatically or via the UI.

## Step 2: Create Delta Sharing Shares

Each domain team creates a **Share** containing their gold data products. Shares are the underlying delivery mechanism that Private Exchange listings use to securely distribute data to consumers.

**One share per data product** — this gives fine-grained control over what each listing exposes.

In [0]:
%sql
-- ============================================================
-- SHARE 1: ADOIT Application Landscape (EA Domain)
-- ============================================================
CREATE SHARE IF NOT EXISTS adoit_application_landscape 
COMMENT 'Application Landscape data product | Domain: Enterprise Architecture | Owner: EA Team | SLA: 4h freshness | Quality: >95%';

ALTER SHARE adoit_application_landscape 
ADD TABLE adoit_product.gold.application_landscape 
COMMENT 'Gold-tier: Application portfolio with business capabilities, criticality, lifecycle status';

-- ============================================================
-- SHARE 2: SNOW Service Health (ITSM Domain)
-- ============================================================
CREATE SHARE IF NOT EXISTS snow_service_health 
COMMENT 'Service Health data product | Domain: IT Service Management | Owner: ITSM Team | SLA: 2h freshness | Quality: >90%';

ALTER SHARE snow_service_health 
ADD TABLE snow_product.gold.service_health 
COMMENT 'Gold-tier: Incident metrics, change success rates, SLA compliance per application';

-- ============================================================
-- SHARE 3: Cross-Domain Application Risk Profile (Platform)
-- ============================================================
CREATE SHARE IF NOT EXISTS datamesh_application_risk_profile 
COMMENT 'Application Risk Profile data product | Domain: Cross-Domain | Owner: Platform Team | SLA: 6h freshness | Quality: >90%';

ALTER SHARE datamesh_application_risk_profile 
ADD TABLE data_mesh_hub.cross_domain.application_risk_profile 
COMMENT 'Gold-tier: Composite risk scoring combining EA criticality with ITSM operational health';

In [0]:
%sql
-- List all shares in this metastore
SHOW SHARES;

-- Inspect the ADOIT share contents
-- DESCRIBE SHARE adoit_application_landscape;

## Step 3: Create Recipients

Recipients represent **consuming organizations or metastores**. In a Data Mesh context:
- Each business unit's metastore is a **recipient**
- For **same-metastore sharing**, recipients aren't needed — Unity Catalog handles access directly
- For **cross-metastore** or **cross-region** sharing (common in large enterprises), you create recipients using the consumer's **sharing identifier**

### How to Get a Sharing Identifier
On the **consumer's workspace**, run:
```sql
SELECT current_metastore() AS metastore_id;
```
Then find the sharing identifier in **Account Console → Data → Metastores → [metastore name] → Sharing**.

In [0]:
%sql
-- ============================================================
-- Recipients for internal business units
-- Replace sharing identifiers with actual values from consumer workspaces
-- ============================================================

-- To get a metastore's sharing identifier, run on the CONSUMER workspace:
--   SELECT current_metastore() AS metastore_id;
-- Then find the sharing identifier in Account Console → Metastores

-- Example recipients (uncomment and replace with real identifiers):
-- CREATE RECIPIENT IF NOT EXISTS cto_office 
--   USING ID '<sharing-identifier-from-cto-workspace>'
--   COMMENT 'CTO Office | Consumers: Application risk oversight';

-- CREATE RECIPIENT IF NOT EXISTS risk_committee
--   USING ID '<sharing-identifier-from-risk-workspace>'
--   COMMENT 'Risk Committee | Consumers: SLA compliance monitoring';

-- CREATE RECIPIENT IF NOT EXISTS compliance_team
--   USING ID '<sharing-identifier-from-compliance-workspace>'
--   COMMENT 'Compliance Team | Consumers: Audit and regulatory reporting';

-- Grant share access to recipients:
-- GRANT SELECT ON SHARE adoit_application_landscape TO RECIPIENT cto_office;
-- GRANT SELECT ON SHARE snow_service_health TO RECIPIENT risk_committee;
-- GRANT SELECT ON SHARE datamesh_application_risk_profile TO RECIPIENT cto_office;
-- GRANT SELECT ON SHARE datamesh_application_risk_profile TO RECIPIENT risk_committee;

-- For demo: list what's in each share
SHOW ALL IN SHARE adoit_application_landscape;

## Step 4: Create Private Listings (Provider Console UI)

Listings wrap Delta Sharing shares into **discoverable products** with rich metadata, documentation, and sample queries. This is done in the **Provider Console UI**.

### Procedure
1. **Marketplace → Provider Console → Listings tab → Create listing**
2. **Listing type**: Select "Data" (for table-based products)
3. **Share**: Select the share created in Step 2
4. **Visibility**: Choose **"Private exchange"** (NOT public) and select the exchange
5. **Fill in listing metadata** (see table below)

### Listing Configurations

| Field | Application Landscape | Service Health | Application Risk Profile |
|---|---|---|---|
| **Name** | Application Landscape | Service Health | Application Risk Profile |
| **Subtitle** | Enterprise application portfolio with business capability mapping | IT service operational health metrics and SLA compliance | Cross-domain composite risk scoring |
| **Category** | Enterprise Architecture | IT Operations | Risk & Compliance |
| **Domain** | Enterprise Architecture | IT Service Management | Cross-Domain |
| **Owner** | EA Team | ITSM Team | Platform Team |
| **SLA** | 4-hour freshness | 2-hour freshness | 6-hour freshness |
| **Sample Query** | `SELECT app_name, criticality, capability_count FROM ...` | `SELECT affected_application_id, sla_compliance_pct FROM ...` | `SELECT app_name, composite_risk_score, risk_factors FROM ...` |
| **Use Cases** | App rationalization, investment planning, cloud migration | Incident trending, SLA monitoring, change success analysis | Executive risk dashboards, audit reporting, portfolio health |
| **Documentation** | Link to EA data dictionary | Link to ITSM data dictionary | Link to cross-domain lineage docs |

> 💡 **Tip**: Include sample SQL queries in the listing description — consumers can copy-paste them immediately after installing the product.

## Step 5: Create the Private Exchange

The Private Exchange is the **storefront** — an invite-only marketplace where members can browse, request, and install data products. Listings added here are **never visible on the public Databricks Marketplace**.

### Procedure
1. **Provider Console → Exchanges tab → Create exchange**
2. **Name**: `Enterprise Data Mesh Exchange`
3. **Add members**: For each consuming business unit, click "Add member" and enter their sharing identifier
4. **Add listings**: Assign the 3 private listings from Step 4

### Exchange Structure

```
Enterprise Data Mesh Exchange (Private)
│
├── 👥 Members
│   ├── CTO Office         (metastore: xxx-xxx-xxx)
│   ├── Risk Committee     (metastore: yyy-yyy-yyy)
│   └── Compliance Team    (metastore: zzz-zzz-zzz)
│
└── 📦 Listings
    ├── Application Landscape      (EA Domain → adoit_application_landscape share)
    ├── Service Health             (ITSM Domain → snow_service_health share)
    └── Application Risk Profile   (Cross-Domain → datamesh_application_risk_profile share)
```

> 🔒 **Security**: Only invited members can see these listings. You can add/remove members and listings at any time. Each member must accept the exchange invitation before browsing.

## Step 6: Consumer Experience

Once the exchange is live, this is what **consumer teams** see and do:

### Consumer Workflow
1. **Open Marketplace** — Click the Marketplace icon in the sidebar of their workspace
2. **Filter by "Private exchange"** — Check the Private exchange filter on the Marketplace home page to see only internal listings
3. **Browse listings** — Each listing shows the description, sample queries, documentation links, SLA, and owner contact
4. **Request or Get access**:
   - **Instant access** (free listings): Click **"Get"** → tables are immediately available
   - **Approval-based**: Click **"Request access"** → provider approves → tables become available
5. **Install into catalog** — Once approved, the data product appears as a **shared catalog** in the consumer's Unity Catalog
6. **Query like any local table** — Standard SQL works immediately:
   ```sql
   SELECT app_name, composite_risk_score, risk_factors
   FROM <shared_catalog>.cross_domain.application_risk_profile
   WHERE composite_risk_score IN ('CRITICAL', 'HIGH');
   ```

### What Consumers Get
- Full **Unity Catalog governance** on installed products: lineage tracking, audit logs, access controls
- **Automatic updates** — when the provider refreshes the gold table, consumers see the latest data (no re-installation needed)
- **Schema visibility** — column names, types, and comments are preserved
- **Change Data Feed** — if enabled, consumers can read incremental changes

> 📌 **Note**: Consumers do NOT get write access. The data product remains read-only, owned by the domain team.

In [0]:
%sql
-- ============================================================
-- Audit: Review access grants on each share
-- ============================================================

-- Who has access to the ADOIT Application Landscape?
SHOW GRANTS ON SHARE adoit_application_landscape;

-- Who has access to the SNOW Service Health?
-- SHOW GRANTS ON SHARE snow_service_health;

-- Who has access to the Cross-Domain Risk Profile?
-- SHOW GRANTS ON SHARE datamesh_application_risk_profile;

## Step 7: Update Data Product Registry

Extend the existing governance tables (from notebook 07) to track **exchange and listing status** for each data product. This closes the loop between the governance layer and the distribution layer.

In [0]:
%sql
-- ============================================================
-- Add exchange tracking columns to existing registry
-- ============================================================
ALTER TABLE data_mesh_hub.platform.data_product_registry 
  ADD COLUMNS IF NOT EXISTS (
    share_name STRING COMMENT 'Delta Sharing share name',
    listing_name STRING COMMENT 'Marketplace listing name',
    exchange_name STRING COMMENT 'Private exchange name',
    listing_status STRING COMMENT 'Draft, Published, Archived'
  );

-- ============================================================
-- Update existing products with exchange info
-- ============================================================
UPDATE data_mesh_hub.platform.data_product_registry
SET share_name = 'adoit_application_landscape',
    listing_name = 'Application Landscape',
    exchange_name = 'Enterprise Data Mesh Exchange',
    listing_status = 'Published'
WHERE product_name = 'Application Landscape';

UPDATE data_mesh_hub.platform.data_product_registry
SET share_name = 'snow_service_health',
    listing_name = 'Service Health',
    exchange_name = 'Enterprise Data Mesh Exchange',
    listing_status = 'Published'
WHERE product_name = 'Service Health';

UPDATE data_mesh_hub.platform.data_product_registry
SET share_name = 'datamesh_application_risk_profile',
    listing_name = 'Application Risk Profile',
    exchange_name = 'Enterprise Data Mesh Exchange',
    listing_status = 'Published'
WHERE product_name = 'Application Risk Profile';

In [0]:
%sql
-- Verify the registry now includes exchange metadata
SELECT product_name, domain, status, share_name, listing_name, exchange_name, listing_status
FROM data_mesh_hub.platform.data_product_registry
ORDER BY product_id;

## Summary: Before vs After Private Exchange

| Aspect | Before (Direct UC Grants) | After (Private Exchange) |
|---|---|---|
| **Discovery** | Manual — consumers must know catalog/schema names | Marketplace storefront with search, categories, descriptions |
| **Access Request** | Admin runs `GRANT` SQL manually | Self-service: consumer clicks "Request" in Marketplace |
| **Documentation** | `TBLPROPERTIES` and `COMMENT` only | Rich listing page: descriptions, sample queries, use cases |
| **Onboarding** | Share catalog name + grant access per table | Invite to exchange → consumer browses and self-serves |
| **Audit** | UC audit logs only | UC audit + Marketplace request tracking + share audit |
| **Cross-Org Sharing** | Not possible across metastores | Delta Sharing enables cross-metastore, cross-region |
| **Lifecycle Mgmt** | No formal publish/deprecate workflow | Draft → Published → Archived with listing status tracking |
| **Governance** | Table-level grants + TBLPROPERTIES | All of the above + data contracts + exchange membership control |

---

## Next Steps
1. **Run notebooks 00–08** first to set up the base Data Mesh infrastructure (domain catalogs, gold tables, governance tables, Genie space, dashboard)
2. **Run this notebook (09)** to create shares and extend the registry with exchange metadata
3. **Complete the UI steps** (Steps 1, 4, 5) in the Provider Console to set up the provider profile, listings, and exchange
4. **Invite consumer teams** by adding their metastore sharing identifiers as exchange members
5. **Monitor on the dashboard** — [Data Mesh — Product Health Dashboard](#dashboard-4284494353825727) tracks product health, and the updated registry now includes listing status

### Related Notebooks
- [00_Data_Mesh_Overview_and_Setup](#notebook-101624548147115) — Infrastructure & catalog setup
- [07_Data_Product_Governance](#notebook-1497137401392699) — Registry, contracts, observability
- [08_Advanced_Governance](#notebook-1497137401392702) — CDF, row-level security, alerting